In [ ]:
!pip install git+https://github.com/google-research/bleurt.git

  Cloning https://github.com/google-research/bleurt.git to /tmp/pip-req-build-x3l97h1l
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/bleurt.git /tmp/pip-req-build-x3l97h1l
  Resolved https://github.com/google-research/bleurt.git to commit cebe7e6f996b40910cfaa520a63db47807e3bf5c
  Preparing metadata (setup.py) ... done
  Created wheel for BLEURT: filename=BLEURT-0.0.2-py3-none-any.whl size=16456766 sha256=d7e7a66010c76d15174e20f646608f9643431ac88691929d0886c7ec7f317e55
  Stored in directory: /tmp/pip-ephem-wheel-cache-6z8tx01l/wheels/30/af/34/e148007788b060e4c76e7ecf68e70c692dff0f2632e62ac454
Successfully built BLEURT


In [ ]:
!pip install bert-score rouge-score sacrebleu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
 

In [ ]:
!pip install --upgrade pip
!pip install transformers datasets peft bitsandbytes accelerate evaluate
!pip install -q rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]


In [ ]:
import os, json, random, argparse, logging, datetime
from dataclasses import dataclass, asdict
from typing import Dict, List, Any, Optional, Tuple
import torch
from torch.utils.data import Dataset
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType
)
import evaluate
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer as HFTok
import numpy as np
import inspect
from transformers.utils import is_flash_attn_2_available
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, EarlyStoppingCallback, set_seed,
    AutoModelForSequenceClassification
)
import bitsandbytes as bnb
from google.colab import files
uploaded = files.upload()

Saving BioASQ-training13b.zip to BioASQ-training13b.zip


In [ ]:
zip_name = list(uploaded.keys())[0]
os.makedirs("/content/data/bioasq", exist_ok=True)
!unzip -q "$zip_name" -d /content/data/bioasq
!ls -l /content/data/bioasq | head

total 4
drwxrwxr-x 2 root root 4096 Oct  7  2024 BioASQ-training13b


In [ ]:
# Reduce fragmentation
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

transformers.logging.set_verbosity_error()

# Collator
class CausalLMPadCollator:
    def __init__(self, pad_id: int, label_pad_id: int = -100):
        self.pad_id = pad_id
        self.label_pad_id = label_pad_id

    def __call__(self, feats):
        L = max(len(f["input_ids"]) for f in feats)
        def pad(x, v): return x + [v] * (L - len(x))
        return {
            "input_ids":      torch.tensor([pad(f["input_ids"], self.pad_id)      for f in feats]),
            "attention_mask": torch.tensor([pad(f["attention_mask"], 0)           for f in feats]),
            "labels":         torch.tensor([pad(f["labels"], self.label_pad_id)   for f in feats]),
        }

In [ ]:
# Config
@dataclass
class OptimizedFTConfig:
    model_name: str = "meta-llama/Llama-3.2-11B-Vision-Instruct"
    data_path: str = "data/bioasq/BioASQ-training13b/training13b.json"
    output_dir: str = "./outputs/bioasq_rag_optimized"
    adapter_name: str = "lora-bioasq-rag"

    # Tokenization
    max_length: int = 768
    max_new_tokens: int = 256
    max_passages: int = 5
    max_passage_length: int = 150

    # Data
    train_frac: float = 0.8
    val_frac: float = 0.1
    seed: int = 42
    add_negative_examples: bool = True
    negative_ratio: float = 0.2

    # Train
    per_device_train_bs: int = 1
    per_device_eval_bs: int = 1
    grad_accum: int = 2
    lr: float = 5e-5
    lr_scheduler_type: str = "cosine"
    num_epochs: int = 3
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 0.5

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    target_modules: tuple = ("q_proj", "k_proj", "v_proj", "o_proj")  # attn-only for lower mem
    lora_plus: bool = True
    lora_plus_lr_ratio: float = 16.0

    # System
    gradient_checkpointing: bool = True
    use_flash_attention: bool = True
    optim: str = "paged_adamw_8bit"
    tf32: bool = True
    bf16: bool = True

    # Eval / logging
    eval_steps: int = 200
    save_steps: int = 200
    logging_steps: int = 50
    save_total_limit: int = 3
    load_best_model_at_end: bool = True
    early_stop_patience: int = 5

    compute_factual_consistency: bool = True
    n_eval_samples: int = 300

In [ ]:
# Data processing
class BioASQRAGProcessor:
    INSUFFICIENT = "The passages do not contain sufficient information to answer."

    def __init__(self, cfg: OptimizedFTConfig):
        self.cfg = cfg
        self.logger = logging.getLogger(__name__)

    def load_data(self, path: str) -> List[Dict[str, Any]]:
        with open(path) as f:
            data = json.load(f)
        qs = data.get("questions", data)
        self.logger.info(f"Loaded {len(qs)} questions")
        return qs

    def normalize_example(self, q: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        ideal = q.get("ideal_answer", "")
        if isinstance(ideal, list):
            ideal = " ".join(t.strip() for t in ideal if t.strip())
        ideal = ideal.strip()
        if (q.get("type", "").lower() == "yesno" or
            ideal.lower() in {"yes", "no"} or
            len(ideal.split()) < 10):
            return None
        if not q.get("body") or not ideal:
            return None

        snippets = []
        for s in q.get("snippets", []):
            text = s.get("text", "").strip()
            if text and len(text.split()) > 5:
                snippets.append({"text": text, "document": s.get("document", "")})
        if not snippets:
            return None

        return {
            "id": q.get("id", ""),
            "question": q["body"].strip(),
            "answer": ideal,
            "snippets": snippets,
            "type": q.get("type", "summary"),
        }

    def format_passages(self, snippets: List[Dict], scores: Optional[List[float]] = None) -> str:
        cfg = self.cfg
        if scores:
            paired = sorted(zip(snippets, scores), key=lambda x: x[1], reverse=True)
            snippets = [s for s, _ in paired[:cfg.max_passages]]
        else:
            snippets = snippets[:cfg.max_passages]

        formatted = []
        for i, sn in enumerate(snippets):
            text = sn["text"]
            words = text.split()
            if len(words) > cfg.max_passage_length:
                text = " ".join(words[:cfg.max_passage_length]) + "..."
            formatted.append(f"[Passage {i+1}]: {text}")
        return "\n\n".join(formatted)

    def create_negative_example(self, ex: Dict, all_data: List[Dict]) -> Dict:
        relevant = ex["snippets"][:3]
        others = [q for q in all_data if q["id"] != ex["id"]]
        irr = []
        for _ in range(2):
            oq = random.choice(others)
            if oq["snippets"]:
                irr.append(random.choice(oq["snippets"]))
        all_snips = relevant + irr
        random.shuffle(all_snips)
        return {**ex, "snippets": all_snips, "has_irrelevant": True, "answer": self.INSUFFICIENT}

In [ ]:
class RAGTokenizer:
    def __init__(self, tokenizer: AutoTokenizer, cfg: OptimizedFTConfig):
        self.tok = tokenizer
        self.cfg = cfg
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.tok.padding_side = "left"

    def build_prompt(self, ex: Dict[str, Any]) -> str:
        system = (
            "You are a knowledgeable biomedical QA assistant. "
            "Answer the question using ONLY information from the provided passages. "
            "Be factually accurate and concise. If the passages don't contain the answer, say so."
        )
        context = BioASQRAGProcessor(self.cfg).format_passages(ex["snippets"])
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": f"Question: {ex['question']}\n\nRelevant passages:\n{context}"},
        ]
        return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def tokenize_example(self, ex: Dict[str, Any]) -> Dict[str, Any]:
        prompt = self.build_prompt(ex)
        prompt_ids = self.tok(prompt, add_special_tokens=True, truncation=True,
                              max_length=self.cfg.max_length)["input_ids"]
        target_ids = self.tok(ex["answer"] + self.tok.eos_token,
                              add_special_tokens=False)["input_ids"]

        total_max = self.cfg.max_length
        avail_prompt = total_max - len(target_ids)
        if avail_prompt < 16:
            avail_prompt = 16
            target_ids = target_ids[: total_max - avail_prompt]

        prompt_ids = prompt_ids[-avail_prompt:]
        input_ids = prompt_ids + target_ids
        attn = [1] * len(input_ids)
        ans_start = len(prompt_ids)
        labels = [-100] * ans_start + input_ids[ans_start:]

        return {
            "input_ids": input_ids,
            "attention_mask": attn,
            "labels": labels,
            "length": len(input_ids),
            "has_irrelevant": ex.get("has_irrelevant", False),
        }

In [ ]:
#Evaluator
class RAGEvaluator:
    def __init__(self, cfg: OptimizedFTConfig):
        self.cfg = cfg
        self.rouge = evaluate.load("rouge")
        self.bert = evaluate.load("bertscore")
        self.bleu = evaluate.load("sacrebleu")
        self.nli_tok = None
        self.nli_model = None
        if cfg.compute_factual_consistency:
            try:
                self.nli_tok = AutoTokenizer.from_pretrained("microsoft/deberta-large-mnli")
                self.nli_model = AutoModelForSequenceClassification.from_pretrained(
                    "microsoft/deberta-large-mnli"
                ).to("cpu").eval()
            except Exception:
                logging.warning("MNLI model not available; skipping factual consistency.")

    def compute_all(self, preds: List[str], refs: List[str], ctxs: Optional[List[str]] = None) -> Dict[str, float]:
        out = {}
        r = self.rouge.compute(predictions=preds, references=refs)
        out.update({f"rouge_{k}": float(v) for k, v in r.items()})
        b = self.bert.compute(predictions=preds, references=refs, lang="en")
        out["bertscore_f1"] = float(np.mean(b["f1"]))
        bl = self.bleu.compute(predictions=preds, references=[[r] for r in refs])
        out["bleu"] = float(bl["score"])

        if ctxs and self.nli_tok and self.nli_model:
            scores = []
            for pred, ctx in zip(preds, ctxs):
                enc = self.nli_tok(ctx[:1500], pred[:512], return_tensors="pt", truncation=True)
                with torch.no_grad():
                    probs = self.nli_model(**enc).logits.softmax(-1)[0]
                    scores.append(float(probs[2]))  # entailment
            out["factual_consistency"] = float(np.mean(scores))

        pred_len = [len(p.split()) for p in preds]
        ref_len = [len(r.split()) for r in refs]
        out["avg_pred_length"] = float(np.mean(pred_len))
        out["length_ratio"] = float(np.mean(pred_len) / max(1, np.mean(ref_len)))
        return out

In [ ]:
# LoRA+ Trainer
class LoRAPlusTrainer(Trainer):
    def __init__(self, lora_plus_lr_ratio=16.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lora_plus_lr_ratio = lora_plus_lr_ratio

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        lora_a, lora_b, other = [], [], []
        for name, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            if "lora_A" in name:
                lora_a.append(p)
            elif "lora_B" in name:
                lora_b.append(p)
            else:
                other.append(p)

        groups = []
        if lora_a:
            groups.append({"params": lora_a, "lr": self.args.learning_rate, "weight_decay": self.args.weight_decay})
        if lora_b:
            groups.append({"params": lora_b, "lr": self.args.learning_rate * self.lora_plus_lr_ratio,
                           "weight_decay": self.args.weight_decay})
        if other:
            groups.append({"params": other, "lr": self.args.learning_rate, "weight_decay": self.args.weight_decay})

        opt_name = (self.args.optim or "adamw_torch").lower()
        if opt_name in {"paged_adamw_8bit", "adamw_bnb_8bit"}:
            opt_cls = bnb.optim.PagedAdamW8bit
            opt_kwargs = dict(
                lr=self.args.learning_rate,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
                weight_decay=self.args.weight_decay,
            )
        elif opt_name in {"adamw_torch", "adamw_torch_fused", "adamw_torch_xla"}:
            opt_cls = torch.optim.AdamW
            fused_supported = "fused" in torch.optim.AdamW.__init__.__code__.co_varnames
            opt_kwargs = dict(
                lr=self.args.learning_rate,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
                weight_decay=self.args.weight_decay,
                **({"fused": True} if fused_supported and torch.cuda.is_available() else {}),
            )
        elif opt_name == "adafactor":
            from transformers.optimization import Adafactor
            opt_cls = Adafactor
            opt_kwargs = dict(
                lr=self.args.learning_rate, eps=(1e-30, 1e-3), clip_threshold=1.0,
                decay_rate=-0.8, beta1=None, weight_decay=self.args.weight_decay,
                scale_parameter=False, relative_step=False, warmup_init=False,
            )
        else:
            raise ValueError(f"Unknown optimizer '{self.args.optim}'. Use 'adamw_torch' or 'paged_adamw_8bit'.")

        self.optimizer = opt_cls(groups, **opt_kwargs)
        return self.optimizer

In [ ]:
#Main
def main(cfg: OptimizedFTConfig):
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(name)s - %(message)s")
    logger = logging.getLogger("train")
    set_seed(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)
    json.dump(asdict(cfg), open(os.path.join(cfg.output_dir, "config.json"), "w"), indent=2)

    # Data
    proc = BioASQRAGProcessor(cfg)
    raw = proc.load_data(cfg.data_path)
    norm = [proc.normalize_example(q) for q in raw]
    data = [d for d in norm if d]
    if cfg.add_negative_examples:
        nneg = int(len(data) * cfg.negative_ratio)
        for _ in range(nneg):
            data.append(proc.create_negative_example(random.choice(data), data))
        logger.info(f"Added {nneg} negative examples.")
    random.shuffle(data)
    n = len(data); n_train = int(cfg.train_frac * n); n_val = int(cfg.val_frac * n)
    train_data, val_data, test_data = data[:n_train], data[n_train:n_train+n_val], data[n_train+n_val:]
    logger.info(f"Splits — train {len(train_data)} | val {len(val_data)} | test {len(test_data)}")

    # Tokenizer / model
    tok = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True, use_fast=True)
    rag_tok = RAGTokenizer(tok, cfg)


    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if cfg.bf16 else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model_kwargs = {
        "torch_dtype": torch.bfloat16 if cfg.bf16 else torch.float16,
        "device_map": {"": 0},
        "trust_remote_code": True,
        "attn_implementation": "flash_attention_2" if (cfg.use_flash_attention and is_flash_attn_2_available())
                               else "sdpa",
    }
    model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)

    torch.backends.cuda.matmul.allow_tf32 = cfg.tf32
    torch.backends.cudnn.allow_tf32 = cfg.tf32
    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False

    # LoRA
    lora_cfg = LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules), bias="none", task_type=TaskType.CAUSAL_LM,
    )


    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Tokenize
    def tok_split(split):
        return [rag_tok.tokenize_example(x) for x in split]
    train_tok, val_tok = tok_split(train_data), tok_split(val_data)
    train_ds, val_ds = HFDataset.from_list(train_tok), HFDataset.from_list(val_tok)
    collator = CausalLMPadCollator(pad_id=tok.pad_token_id)

    # Args
    args = TrainingArguments(
        output_dir=cfg.output_dir,
        per_device_train_batch_size=cfg.per_device_train_bs,
        per_device_eval_batch_size=cfg.per_device_eval_bs,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        lr_scheduler_type=cfg.lr_scheduler_type,
        warmup_ratio=cfg.warmup_ratio,
        num_train_epochs=cfg.num_epochs,
        optim=cfg.optim,
        weight_decay=cfg.weight_decay,
        max_grad_norm=cfg.max_grad_norm,
        bf16=cfg.bf16,
        bf16_full_eval=True,
        tf32=cfg.tf32,

        logging_steps=cfg.logging_steps,
        report_to="none",
        eval_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        load_best_model_at_end=cfg.load_best_model_at_end,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        group_by_length=True,
        length_column_name="length",

        prediction_loss_only=True,   #  avoid storing logits
        eval_accumulation_steps=1,
        #predict_with_generate=False,
        seed=cfg.seed,
    )

    callbacks = [EarlyStoppingCallback(cfg.early_stop_patience, early_stopping_threshold=0.001)]

    # Trainer
    common = dict(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator, tokenizer=tok,
        compute_metrics=None, callbacks=callbacks,
    )
    if cfg.lora_plus:
        trainer = LoRAPlusTrainer(lora_plus_lr_ratio=cfg.lora_plus_lr_ratio, **common)
    else:
        trainer = Trainer(**common)

    # Train
    logger.info("Training...")
    train_result = trainer.train()
    json.dump(train_result.metrics, open(os.path.join(cfg.output_dir, "train_results.json"), "w"), indent=2)

    # Save adapters
    adapter_dir = os.path.join(cfg.output_dir, cfg.adapter_name)
    os.makedirs(adapter_dir, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tok.save_pretrained(adapter_dir)
    logger.info(f"Saved LoRA adapter to: {adapter_dir}")

    # save merged full model
    merged_dir = os.path.join(cfg.output_dir, "merged-full-model")
    try:
        merged = model.merge_and_unload()
        os.makedirs(merged_dir, exist_ok=True)
        merged.save_pretrained(merged_dir, safe_serialization=True)
        tok.save_pretrained(merged_dir)
        logger.info(f"Saved merged full model to: {merged_dir}")
    except Exception as e:
        logger.warning(f"merge_and_unload not available; skipping merged save. {e}")

#Base / LoRA comparison
    logger.info("Evaluating LoRA then Base (sequentially, quantized) ...")
    evaluator = RAGEvaluator(cfg)
    N = min(cfg.n_eval_samples, len(test_data))
    subset = random.sample(test_data, N)

    def build_ctx(ex): return proc.format_passages(ex["snippets"])

    @torch.inference_mode()
    def generate_preds(m, examples):
        m.eval()
        preds, refs, ctxs = [], [], []
        for ex in examples:
            prompt = rag_tok.build_prompt(ex)
            t = tok(prompt, return_tensors="pt", truncation=True, max_length=cfg.max_length).to(m.device)
            out = m.generate(
                **t, max_new_tokens=cfg.max_new_tokens,
                do_sample=False, temperature=0.0,
                pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
            )
            text = tok.decode(out[0][t["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            preds.append(text); refs.append(ex["answer"]); ctxs.append(build_ctx(ex))
        return preds, refs, ctxs

    # LoRA metrics (on GPU)
    lora_preds, refs, ctxs = generate_preds(model, subset)
    lora_metrics = evaluator.compute_all(lora_preds, refs, ctxs)

    # Free LoRA model before loading base
    del model
    import gc; gc.collect()
    torch.cuda.empty_cache()

    # Base model, quantized 4-bit to avoid OOM
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        trust_remote_code=True,
        device_map={"": 0},
        quantization_config=bnb_config,  # use the same 4-bit config
        attn_implementation=("flash_attention_2" if (cfg.use_flash_attention and is_flash_attn_2_available()) else "sdpa"),
    )
    base_preds, _, _ = generate_preds(base_model, subset)
    base_metrics = evaluator.compute_all(base_preds, refs, ctxs)

    # free base model
    del base_model; gc.collect(); torch.cuda.empty_cache()

    INSUFF = BioASQRAGProcessor.INSUFFICIENT.lower()
    def abstention(preds, exs):
        neg_ix = [i for i, e in enumerate(exs) if e.get("has_irrelevant", False)]
        if not neg_ix: return 0.0
        hits = sum(1 for i in neg_ix if INSUFF in preds[i].lower())
        return hits / len(neg_ix)

    base_metrics["abstention_neg"] = abstention(base_preds, subset)
    lora_metrics["abstention_neg"] = abstention(lora_preds, subset)


    comp_path = os.path.join(cfg.output_dir, "final_comparison_metrics.json")
    json.dump({"base": base_metrics, "lora": lora_metrics}, open(comp_path, "w"), indent=2)
    logger.info(f"Saved comparison metrics to {comp_path}")

In [ ]:
# CLI
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str)
    parser.add_argument("--model_name", type=str)
    parser.add_argument("--output_dir", type=str)

    import sys
    if any(a.startswith("-f") for a in sys.argv):
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args()

    if args.config and os.path.exists(args.config):
        cfg = OptimizedFTConfig(**json.load(open(args.config)))
    else:
        cfg = OptimizedFTConfig()
    if args.model_name: cfg.model_name = args.model_name
    if args.output_dir: cfg.output_dir = args.output_dir

    main(cfg)

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/89.4k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]